In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from estimation_forecast_functions_var import DataCleaner_JPY
import holidays
from simple_strat_funcs import Clean_Implied_Vols_JPY_with_smile
from scipy.stats import norm
from hedging_strategy_class_NEW_KURT_SKEW_vega_VAR import Compare_Trading_Strategies
import seaborn as sns

/Users/alexvillamartin/Documents/MSc Diss/Code/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Global params

In [2]:
train_size = 0.7
test_size = 1 - train_size

TICKER = "USDJPY"

# always use same amount of money in USD to start and convert where necessary
IC_USD = 1_000_000 # initial capital in USD
current_rate = 147.52
IC_JPY = IC_USD * current_rate
NOTIONAL_USD = 1_000_000 # this is used in strategy as our notional to ensure trade positions are in domestic currency

MAX_DELTA_DIFF = 500 # this param doesnt change much as our delta positions are usually much larger
SIGNAL_LB = 0.01
SIGNAL_UB = 0.99
TRANSACTION_COST_BOOL = True
TRANSACTION_COSTS_SPOT = 0.0002 / 2 # for half a leg ie selling/buying once not to enter and exit the trade
TRANSACTION_COSTS_OPTION = 0.0005 / 2 # [change to vol spreads later] for half a leg again, both in decimals

HYPERPARAM_SORT = 'sharpe_ratio' # can be 'sharpe_ratio', 'cagr', 'max_drawdown', 'total_return'
GARCH_1_2_INDICATOR = False

k_bar_MSM = 8
b_MSM = 2.0
gamma_kbar_MSM = 0.5

M = 300 # figarch lags

CONVERT_USD_INDICATOR = True # convert all portfolio values and metrics to USD for fair comparison - use when USD is not domestic

long_threshs = np.array([1.02, 1.03, 1.04, 1.05, 1.06, 1.07,  1.08, 1.09, 1.10, 1.11, 1.12, 1.13, 1.14, 1.15, 1.16, 1.17, 1.18, 1.19, 1.20, 1.21, 1.22, 1.23, 1.24, 1.25, 1.26, 1.27])
short_threshs = np.array([0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92,  0.91, 0.90, 0.89, 0.88, 0.87, 0.86, 0.85, 0.84, 0.83, 0.82, 0.81, 0.80, 0.79, 0.78, 0.77, 0.76, 0.75, 0.74, 0.73])
sig_multipliers = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 20])

# Global data

In [3]:
usdjpy_5m = pd.read_parquet("USD_JPY_5M.parquet").copy()[['c']]
usdjpy_5m.rename(columns={'c': 'spot'}, inplace=True)  
cleaner_usdjpy = DataCleaner_JPY(usdjpy_5m, start_hr=2, end_hr=18, unit_test=False) 
realised_variance_usdjpy = cleaner_usdjpy.clean_data()
daily_log_returns_usdjpy = pd.read_parquet("DF_D_USDJPY.parquet")['Log_r']
start_date = pd.to_datetime(realised_variance_usdjpy.index.min())
end_date = pd.to_datetime(realised_variance_usdjpy.index.max())
spot_curr = pd.read_parquet("/Users/alexvillamartin/Documents/MSc Diss/Code/DF_D_USDJPY.parquet")
overnight_domestic_rate = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/tona_daily.csv").set_index("Date") 
overnight_domestic_rate.index = (pd.to_datetime(overnight_domestic_rate.index, format='mixed').normalize())
overnight_foreign_rate = pd.read_csv("SOFR_daily.csv").set_index("date")

# Global functions

Change holidays where needed. 

In [4]:
def align_spots(df, aligned_df, start, end):

    df = df[(df.index >= start) & (df.index <= end)]

    years = range(start.year, end.year + 1)

    first = holidays.US(years=years)
    second = holidays.Japan(years=years)

    hols = pd.to_datetime(list(set(first) | set(second))).normalize()

    idx  = df.index
    mask = ~idx.isin(hols)
    new_df = df.loc[mask]

    df_aligned_dates = aligned_df.index
    common_dates = new_df.index.intersection(df_aligned_dates)
    new_df = new_df.loc[common_dates]

    return new_df

def sort_rates(r_b, r_t, overn_r, aligned_df, overn_for_r):

    r_b.index = pd.to_datetime(r_b.index)
    valid_mask = ~( 
                   r_t.index.isna())
    r_t_clean = r_t[valid_mask]
    r_t_clean.index = pd.to_datetime(r_t_clean.index, format='%Y-%m-%d')
    
    valid_mask2 = ~(
                   overn_r.index.isna())
    overn_r_clean = overn_r[valid_mask2]
    overn_r_clean.index = pd.to_datetime(overn_r_clean.index, format='%Y-%m-%d')

    valid_mask3 = ~(
                   overn_for_r.index.isna())
    overn_for_r_clean = overn_for_r[valid_mask3]
    overn_for_r_clean.index = pd.to_datetime(overn_for_r_clean.index, format='mixed').normalize()    

    r_b = r_b.reindex(aligned_df.index)
    r_t_clean = r_t_clean.reindex(aligned_df.index)
    overn_r_clean = overn_r_clean.reindex(aligned_df.index)
    overn_for_r_clean = overn_for_r_clean.reindex(aligned_df.index)

    r_b = r_b.ffill()
    r_t_clean = r_t_clean.ffill()
    overn_r_clean = overn_r_clean.ffill()
    overn_for_r_clean = overn_for_r_clean.ffill()

    # convert into decimals and continously compunded versions for BSE
    r_b = pd.to_numeric(r_b['rate_pct'], errors='coerce')
    r_t_clean = pd.to_numeric(r_t_clean['rate_pct'], errors='coerce')
    overn_r_clean = pd.to_numeric(overn_r_clean['tona_pct'], errors='coerce')
    r_b = r_b / 100
    r_t_clean = r_t_clean / 100
    overn_r_clean = overn_r_clean / 100 # not continously compounded
    overn_r_clean *= 1/365 # daily

    overn_for_r_clean = pd.to_numeric(overn_for_r_clean['value'], errors='coerce')
    overn_for_r_clean = overn_for_r_clean / 100 # not continously compounded
    overn_for_r_clean *= 1/365 # daily

    r_b = np.log(1 + r_b)
    r_t_clean = np.log(1 + r_t_clean)

    return r_b, r_t_clean, overn_r_clean, overn_for_r_clean

# H=30, T=1Mo

In [5]:
OPTION_MATURITY_1 = 1 / 12
H_1 = 30 

implied_vol_data_1 = pd.read_excel('/Users/alexvillamartin/Documents/MSc Diss/Code/USDJPY_1MO_ATM_D.xlsx')
vol_smile_data_1 = pd.read_csv("usdjpy_vol_smile_1mo_extra.csv").set_index("CalculationDate")
Data_clean_1 = Clean_Implied_Vols_JPY_with_smile(data=implied_vol_data_1, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdjpy, 
                                align_df2=daily_log_returns_usdjpy, 
                                smile_df=vol_smile_data_1)
implied_vol_data_1, realised_variance_1, daily_log_returns_1, vol_smile_data_1 = Data_clean_1.get_clean_data()

N_1 = len(daily_log_returns_1)
test_align_1 = daily_log_returns_1.iloc[N_1//2:-H_1]
spot_curr_test_1 = align_spots(spot_curr, test_align_1, start_date, end_date)

r_b_1 = pd.read_csv("SOFR_1mo_compounded.csv").set_index("date")
r_t_1 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/tona_1mo_compounded.csv").set_index("Date")[['rate_pct']]

r_b_test_1, r_t_test_1, overn_dom_r_test_1, overn_for_r_test_1= sort_rates(r_b_1, r_t_1, overnight_domestic_rate, test_align_1, overnight_foreign_rate) 
r_b_test_1.ffill(inplace=True)

In [6]:
strategy_1 = Compare_Trading_Strategies(
    return_series=daily_log_returns_1, 
    realised_variance_series=realised_variance_1,
    atm_implied_vol_data=implied_vol_data_1,
    vol_smile_data=vol_smile_data_1,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_JPY,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_1,
    forecast_horizon=H_1,
    spot_series=spot_curr_test_1,
    overnight_domestic_rate=overn_dom_r_test_1,
    overnight_foreign_rate=overn_for_r_test_1,
    domestic_rate=r_t_test_1,
    foreign_rate=r_b_test_1, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_1.prepare_universal_series()
strategy_1.get_BMSM_data()
strategy_1.get_GARCH_data()
strategy_1.get_FIGARCH_data()


Estimated parameters: m0=1.324502e+00, sigma_bar=5.592794e-01
Final log-likelihood: -8.254789e+02
Estimated parameters: m0=1.314606e+00, sigma_bar=5.547390e-01
Final log-likelihood: -1.380087e+03
Estimated parameters: omega=0.0037585331153215874, alpha=0.0672, beta=0.9232
Estimated parameters: omega=0.0024381607944387444, alpha=0.0601, beta=0.9340
Estimated parameters: omega=0.07868108445557308, d=0.2129, beta=0.0485
Final log-likelihood = 398.4028
Estimated parameters: omega=0.05459404140392503, d=0.2697, beta=0.1474
Final log-likelihood = 690.6127


In [7]:
error_metrics_df_1, log_ls, m_z_results, se_results = strategy_1.in_sample_predictions()

In [8]:
error_metrics_df_1

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.541366,NaN,NaN,0.642532,NaN,NaN
BMSM OLS,0.533424,-0.189334,0.424932,0.672640,1.104911,0.865286
GARCH,0.630890,NaN,NaN,0.656541,NaN,NaN
GARCH OLS,0.510494,-1.157352,0.123684,0.648891,-0.152449,0.439430
FIGARCH,0.611234,NaN,NaN,0.722949,NaN,NaN
FIGARCH OLS,0.701454,0.702648,0.758792,0.791459,0.968507,0.833503


In [9]:
log_ls

{'BMSM': np.float64(-825.4788664455974),
 'GARCH': np.float64(422.1766834063768),
 'FIGARCH': np.float64(398.4027651423048)}

In [10]:
m_z_results

{'BMSM': {'alpha_hat': -0.0008428014188223312,
  'beta_hat': 1.181028941191611,
  'alpha_p': 0.1668275192180393,
  'beta_p': 0.016696020269471198},
 'GARCH': {'alpha_hat': 0.0026243435253876673,
  'beta_hat': 0.7073459611686737,
  'alpha_p': 0.0007787724226629067,
  'beta_p': 0.0005878156665088639},
 'FIGARCH': {'alpha_hat': -0.002106297653939457,
  'beta_hat': 1.3329865085812023,
  'alpha_p': 0.04810150841864193,
  'beta_p': 0.011666044274955774}}

In [11]:
se_results

{'BMSM': array([0.03016795, 0.06942618]),
 'GARCH': array([0.00141555, 0.00961038, 0.01124412]),
 'FIGARCH': array([0.00939601, 0.02473124, 0.04100287])}

In [12]:
strategy_1.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.1763
Date:                Sat, 30 Aug 2025   Prob (F-statistic):              0.951
Time:                        22:03:54   Log-Likelihood:                -992.76
No. Observations:                1155   AIC:                             1996.
Df Residuals:                    1150   BIC:                             2021.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0936      0.043     -2.187      0.029      -0.177      -0.010
x1             0.0216      0.069      0.314      0.753      -0.113       0.157
x2            -0.0350      0.076     -0.461      0.645      -0.184       0.114
x3             0.0124      0.048      0.261      0.794      -0.081       0.106
x4            -0.0324      0.057     -0.571      0.568      -0.144       0.079
==============================================================================
Omnibus:                      255.155   Durbin-Watson:                   0.058
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              533.323
Skew:                           1.250   Prob(JB):                    1.55e-116
Kurtosis:                       5.197   Cond. No.                         4.80
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [13]:
strategy_1.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.064
Model:                            OLS   Adj. R-squared:                  0.062
Method:                 Least Squares   F-statistic:                     7.193
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           8.74e-05
Time:                        22:03:57   Log-Likelihood:                -999.05
No. Observations:                1155   AIC:                             2006.
Df Residuals:                    1151   BIC:                             2026.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0802      0.043     -1.857      0.063      -0.165       0.004
x1            -0.1581      0.035     -4.552      0.000      -0.226      -0.090
x2             0.0391      0.044      0.886      0.376      -0.047       0.126
x3            -0.0597      0.050     -1.186      0.236      -0.158       0.039
==============================================================================
Omnibus:                      229.339   Durbin-Watson:                   0.061
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              451.016
Skew:                           1.158   Prob(JB):                     1.16e-98
Kurtosis:                       5.002   Cond. No.                         1.63
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [14]:
strategy_1.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.063
Model:                            OLS   Adj. R-squared:                  0.061
Method:                 Least Squares   F-statistic:                     3.585
Date:                Sat, 30 Aug 2025   Prob (F-statistic):             0.0134
Time:                        22:04:06   Log-Likelihood:                -990.28
No. Observations:                1155   AIC:                             1989.
Df Residuals:                    1151   BIC:                             2009.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0906      0.043     -2.106      0.035      -0.175      -0.006
x1             0.0718      0.043      1.653      0.098      -0.013       0.157
x2             0.0504      0.046      1.092      0.275      -0.040       0.141
x3            -0.0721      0.056     -1.278      0.201      -0.183       0.039
==============================================================================
Omnibus:                      283.797   Durbin-Watson:                   0.042
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              600.257
Skew:                           1.387   Prob(JB):                    4.53e-131
Kurtosis:                       5.187   Cond. No.                         2.20
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=91, T=3MO

In [5]:
OPTION_MATURITY_2 = 3 / 12
H_2 = 91 

vol_smile_data_2 = pd.read_csv("usdjpy_vol_smile_3mo_extra.csv").set_index("CalculationDate")
implied_vol_data_2 = pd.DataFrame({
    'Exchange Date': vol_smile_data_2.index, 
    "Bid": vol_smile_data_2['ATM'], 
    "Ask": vol_smile_data_2['ATM'],
    "BidNet": vol_smile_data_2['ATM']})
Data_clean_2 = Clean_Implied_Vols_JPY_with_smile(data=implied_vol_data_2, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdjpy, 
                                align_df2=daily_log_returns_usdjpy, 
                                smile_df=vol_smile_data_2)
implied_vol_data_2, realised_variance_2, daily_log_returns_2, vol_smile_data_2 = Data_clean_2.get_clean_data()

N_2 = len(daily_log_returns_2)
test_align_2 = daily_log_returns_2.iloc[N_2//2:-H_2]
spot_curr_test_2 = align_spots(spot_curr, test_align_2, start_date, end_date)

r_b_2 = pd.read_csv("SOFR_3mo_compounded.csv").set_index("date")
r_t_2 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/tona_3mo_compounded.csv").set_index("Date")[['rate_pct']]

r_b_test_2, r_t_test_2, overn_dom_r_test_2, overn_for_r_test_2= sort_rates(r_b_2, r_t_2, overnight_domestic_rate, test_align_2, overnight_foreign_rate) 
r_b_test_2.ffill(inplace=True)

In [6]:
strategy_2 = Compare_Trading_Strategies(
    return_series=daily_log_returns_2, 
    realised_variance_series=realised_variance_2,
    atm_implied_vol_data=implied_vol_data_2,
    vol_smile_data=vol_smile_data_2,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_JPY,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_2,
    forecast_horizon=H_2,
    spot_series=spot_curr_test_2,
    overnight_domestic_rate=overn_dom_r_test_2,
    overnight_foreign_rate=overn_for_r_test_2,
    domestic_rate=r_t_test_2,
    foreign_rate=r_b_test_2, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_2.prepare_universal_series()
strategy_2.get_BMSM_data()
strategy_2.get_GARCH_data()
strategy_2.get_FIGARCH_data()

Estimated parameters: m0=1.324502e+00, sigma_bar=5.592794e-01
Final log-likelihood: -8.254789e+02
Estimated parameters: m0=1.308990e+00, sigma_bar=5.670236e-01
Final log-likelihood: -1.366077e+03
Estimated parameters: omega=0.0037585331153215874, alpha=0.0672, beta=0.9232
Estimated parameters: omega=0.00229459625455363, alpha=0.0559, beta=0.9387
Estimated parameters: omega=0.07868108445557308, d=0.2129, beta=0.0485
Final log-likelihood = 398.4028
Estimated parameters: omega=0.05667073186735744, d=0.2664, beta=0.1479
Final log-likelihood = 646.8731


In [8]:
error_metrics_df_2, _, m_z_results_2, _ = strategy_2.in_sample_predictions()

In [9]:
error_metrics_df_2

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.615191,NaN,NaN,0.757018,NaN,NaN
BMSM OLS,0.609801,-0.035482,0.485851,0.776925,0.190795,0.575639
GARCH,0.674760,NaN,NaN,0.778089,NaN,NaN
GARCH OLS,0.754867,0.625761,0.734199,0.830819,0.479746,0.684248
FIGARCH,0.716504,NaN,NaN,0.842047,NaN,NaN
FIGARCH OLS,1.242292,1.247868,0.893827,1.106566,1.250909,0.894382


In [10]:
m_z_results_2

{'BMSM': {'alpha_hat': -0.002151209226547242,
  'beta_hat': 1.3716741093695772,
  'alpha_p': 0.011394244628144331,
  'beta_p': 0.0007750533112547495},
 'GARCH': {'alpha_hat': 0.002471559900455196,
  'beta_hat': 0.7077728642436072,
  'alpha_p': 0.009588261161549832,
  'beta_p': 0.0059356728088337165},
 'FIGARCH': {'alpha_hat': -0.0018677138091429853,
  'beta_hat': 1.325410897622434,
  'alpha_p': 0.12956766709881945,
  'beta_p': 0.034638344246914615}}

In [11]:
strategy_2.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.265
Model:                            OLS   Adj. R-squared:                  0.262
Method:                 Least Squares   F-statistic:                     27.22
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.47e-21
Time:                        22:09:33   Log-Likelihood:                -577.61
No. Observations:                1094   AIC:                             1165.
Df Residuals:                    1089   BIC:                             1190.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0814      0.031     -2.624      0.009      -0.142      -0.021
x1             0.2422      0.055      4.430      0.000       0.135       0.349
x2             0.0978      0.052      1.889      0.059      -0.004       0.199
x3            -0.0031      0.035     -0.089      0.929      -0.072       0.066
x4             0.3118      0.051      6.075      0.000       0.211       0.412
==============================================================================
Omnibus:                       25.401   Durbin-Watson:                   0.129
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               22.640
Skew:                           0.295   Prob(JB):                     1.21e-05
Kurtosis:                       2.614   Cond. No.                         5.50
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [12]:
strategy_2.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.112
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     7.156
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           9.26e-05
Time:                        22:09:37   Log-Likelihood:                -692.29
No. Observations:                1094   AIC:                             1393.
Df Residuals:                    1090   BIC:                             1413.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1161      0.035     -3.286      0.001      -0.185      -0.047
x1             0.0491      0.034      1.425      0.154      -0.018       0.117
x2             0.1270      0.036      3.564      0.000       0.057       0.197
x3             0.1574      0.049      3.205      0.001       0.061       0.254
==============================================================================
Omnibus:                       42.840   Durbin-Watson:                   0.047
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               39.586
Skew:                           0.412   Prob(JB):                     2.53e-09
Kurtosis:                       2.563   Cond. No.                         1.81
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [13]:
strategy_2.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.336
Model:                            OLS   Adj. R-squared:                  0.334
Method:                 Least Squares   F-statistic:                     41.65
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.75e-25
Time:                        22:09:40   Log-Likelihood:                -573.50
No. Observations:                1094   AIC:                             1155.
Df Residuals:                    1090   BIC:                             1175.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0541      0.031     -1.732      0.083      -0.115       0.007
x1             0.3189      0.040      8.024      0.000       0.241       0.397
x2             0.1571      0.028      5.522      0.000       0.101       0.213
x3             0.2962      0.054      5.435      0.000       0.189       0.403
==============================================================================
Omnibus:                       42.224   Durbin-Watson:                   0.090
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               28.497
Skew:                           0.276   Prob(JB):                     6.49e-07
Kurtosis:                       2.434   Cond. No.                         2.43
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=182, T=6MO

In [14]:
OPTION_MATURITY_3 = 6 / 12
H_3 = 182

vol_smile_data_3 = pd.read_csv("usdjpy_vol_smile_6mo_extra.csv").set_index("CalculationDate")
implied_vol_data_3 = pd.DataFrame({
    'Exchange Date': vol_smile_data_3.index, 
    "Bid": vol_smile_data_3['ATM'], 
    "Ask": vol_smile_data_3['ATM'],
    "BidNet": vol_smile_data_3['ATM']})
Data_clean_3 = Clean_Implied_Vols_JPY_with_smile(data=implied_vol_data_3, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdjpy, 
                                align_df2=daily_log_returns_usdjpy, 
                                smile_df=vol_smile_data_3)
implied_vol_data_3, realised_variance_3, daily_log_returns_3, vol_smile_data_3 = Data_clean_3.get_clean_data()

N_3 = len(daily_log_returns_3)
test_align_3 = daily_log_returns_3.iloc[N_3//2:-H_3]
spot_curr_test_3 = align_spots(spot_curr, test_align_3, start_date, end_date)

r_b_3 = pd.read_csv("SOFR_6mo_compounded.csv").set_index("date")
r_t_3 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/tona_6mo_compounded.csv").set_index("Date")[['rate_pct']]

r_b_test_3, r_t_test_3, overn_dom_r_test_3, overn_for_r_test_3= sort_rates(r_b_3, r_t_3, overnight_domestic_rate, test_align_3, overnight_foreign_rate) 
r_b_test_3.ffill(inplace=True)

In [15]:
strategy_3 = Compare_Trading_Strategies(
    return_series=daily_log_returns_3, 
    realised_variance_series=realised_variance_3,
    atm_implied_vol_data=implied_vol_data_3,
    vol_smile_data=vol_smile_data_3,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_JPY,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_3,
    forecast_horizon=H_3,
    spot_series=spot_curr_test_3,
    overnight_domestic_rate=overn_dom_r_test_3,
    overnight_foreign_rate=overn_for_r_test_3,
    domestic_rate=r_t_test_3,
    foreign_rate=r_b_test_3, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_3.prepare_universal_series()
strategy_3.get_BMSM_data()
strategy_3.get_GARCH_data()
strategy_3.get_FIGARCH_data()

Estimated parameters: m0=1.324502e+00, sigma_bar=5.592794e-01
Final log-likelihood: -8.254789e+02
Estimated parameters: m0=1.314337e+00, sigma_bar=5.718897e-01
Final log-likelihood: -1.312847e+03
Estimated parameters: omega=0.0037585331153215874, alpha=0.0672, beta=0.9232
Estimated parameters: omega=0.0035718973823386786, alpha=0.0637, beta=0.9301
Estimated parameters: omega=0.07868108445557308, d=0.2129, beta=0.0485
Final log-likelihood = 398.4028
Estimated parameters: omega=0.05575453934837344, d=0.2727, beta=0.1554
Final log-likelihood = 635.9490


In [17]:
error_metrics_df_3, _, m_z_results_3, _ = strategy_3.in_sample_predictions()

In [18]:
error_metrics_df_3

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.763471,NaN,NaN,0.876364,NaN,NaN
BMSM OLS,0.609950,-0.645554,0.259358,0.779994,-0.723623,0.234733
GARCH,0.748402,NaN,NaN,0.869409,NaN,NaN
GARCH OLS,1.008780,0.698698,0.757549,0.850926,-0.135506,0.446119
FIGARCH,0.869674,NaN,NaN,0.956397,NaN,NaN
FIGARCH OLS,1.228257,0.590854,0.722624,1.047008,0.293092,0.615244


In [19]:
m_z_results_3

{'BMSM': {'alpha_hat': -0.0018251626036560947,
  'beta_hat': 1.3897972462341064,
  'alpha_p': 0.17659837241085186,
  'beta_p': 0.01921047826877129},
 'GARCH': {'alpha_hat': 0.002700318519724031,
  'beta_hat': 0.6958576663852237,
  'alpha_p': 0.03812402082628521,
  'beta_p': 0.030124143592779914},
 'FIGARCH': {'alpha_hat': 0.0017876477011378893,
  'beta_hat': 0.916686279620619,
  'alpha_p': 0.26010997159460303,
  'beta_p': 0.6510515809829169}}

In [20]:
strategy_3.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.388
Model:                            OLS   Adj. R-squared:                  0.385
Method:                 Least Squares   F-statistic:                     26.23
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.05e-20
Time:                        22:13:51   Log-Likelihood:                -379.76
No. Observations:                1003   AIC:                             769.5
Df Residuals:                     998   BIC:                             794.1
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0969      0.028     -3.449      0.001      -0.152      -0.042
x1             0.2430      0.050      4.835      0.000       0.144       0.342
x2             0.0735      0.047      1.557      0.119      -0.019       0.166
x3             0.0457      0.034      1.333      0.183      -0.021       0.113
x4             0.2309      0.034      6.802      0.000       0.164       0.297
==============================================================================
Omnibus:                       45.153   Durbin-Watson:                   0.133
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               50.520
Skew:                           0.546   Prob(JB):                     1.07e-11
Kurtosis:                       2.872   Cond. No.                         5.29
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [21]:
strategy_3.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.219
Model:                            OLS   Adj. R-squared:                  0.216
Method:                 Least Squares   F-statistic:                     14.29
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           4.03e-09
Time:                        22:13:55   Log-Likelihood:                -473.49
No. Observations:                1003   AIC:                             955.0
Df Residuals:                     999   BIC:                             974.6
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1888      0.031     -6.010      0.000      -0.250      -0.127
x1             0.1067      0.032      3.356      0.001       0.044       0.169
x2             0.1608      0.033      4.935      0.000       0.097       0.225
x3             0.1162      0.039      2.968      0.003       0.039       0.193
==============================================================================
Omnibus:                      107.619   Durbin-Watson:                   0.052
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               53.105
Skew:                           0.395   Prob(JB):                     2.94e-12
Kurtosis:                       2.196   Cond. No.                         1.61
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [22]:
strategy_3.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.378
Model:                            OLS   Adj. R-squared:                  0.376
Method:                 Least Squares   F-statistic:                     30.66
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           5.82e-19
Time:                        22:13:58   Log-Likelihood:                -469.07
No. Observations:                1003   AIC:                             946.1
Df Residuals:                     999   BIC:                             965.8
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0590      0.031     -1.903      0.057      -0.120       0.002
x1             0.2570      0.036      7.125      0.000       0.186       0.328
x2             0.1973      0.034      5.854      0.000       0.131       0.263
x3             0.2310      0.045      5.184      0.000       0.144       0.318
==============================================================================
Omnibus:                       24.265   Durbin-Watson:                   0.097
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               25.238
Skew:                           0.374   Prob(JB):                     3.31e-06
Kurtosis:                       2.792   Cond. No.                         2.08
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=364, T=1Y

In [23]:
OPTION_MATURITY_4 = 1
H_4 = 364

vol_smile_data_4 = pd.read_csv("usdjpy_vol_smile_1y_extra.csv").set_index("CalculationDate")
implied_vol_data_4 = pd.DataFrame({
    'Exchange Date': vol_smile_data_4.index, 
    "Bid": vol_smile_data_4['ATM'], 
    "Ask": vol_smile_data_4['ATM'],
    "BidNet": vol_smile_data_4['ATM']})
Data_clean_4 = Clean_Implied_Vols_JPY_with_smile(data=implied_vol_data_4, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdjpy, 
                                align_df2=daily_log_returns_usdjpy, 
                                smile_df=vol_smile_data_4)
implied_vol_data_4, realised_variance_4, daily_log_returns_4, vol_smile_data_4 = Data_clean_4.get_clean_data()

N_4 = len(daily_log_returns_4)
test_align_4 = daily_log_returns_4.iloc[N_4//2:-H_4]
spot_curr_test_4 = align_spots(spot_curr, test_align_4, start_date, end_date)

r_b_4 = pd.read_csv("SOFR_1y_compounded.csv").set_index("date")
r_t_4 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/tona_1y_compounded.csv").set_index("Date")[['rate_pct']]

r_b_test_4, r_t_test_4, overn_dom_r_test_4, overn_for_r_test_4= sort_rates(r_b_4, r_t_4, overnight_domestic_rate, test_align_4, overnight_foreign_rate) 
r_b_test_4.ffill(inplace=True)

In [24]:
strategy_4 = Compare_Trading_Strategies(
    return_series=daily_log_returns_4, 
    realised_variance_series=realised_variance_4,
    atm_implied_vol_data=implied_vol_data_4,
    vol_smile_data=vol_smile_data_4,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_JPY,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_4,
    forecast_horizon=H_4,
    spot_series=spot_curr_test_4,
    overnight_domestic_rate=overn_dom_r_test_4,
    overnight_foreign_rate=overn_for_r_test_4,
    domestic_rate=r_t_test_4,
    foreign_rate=r_b_test_4, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_4.prepare_universal_series()
strategy_4.get_BMSM_data()
strategy_4.get_GARCH_data()
strategy_4.get_FIGARCH_data()

Estimated parameters: m0=1.324502e+00, sigma_bar=5.592794e-01
Final log-likelihood: -8.254789e+02
Estimated parameters: m0=1.310971e+00, sigma_bar=5.535291e-01
Final log-likelihood: -1.151542e+03
Estimated parameters: omega=0.0037585331153215874, alpha=0.0672, beta=0.9232
Estimated parameters: omega=0.0024468845943431145, alpha=0.0578, beta=0.9363
Estimated parameters: omega=0.07868108445557308, d=0.2129, beta=0.0485
Final log-likelihood = 398.4028
Estimated parameters: omega=0.060293733896601295, d=0.2454, beta=0.1119
Final log-likelihood = 745.3353


In [25]:
error_metrics_df_4, _, m_z_results_4, _ = strategy_4.in_sample_predictions()

In [26]:
error_metrics_df_4

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.994226,NaN,NaN,0.915472,NaN,NaN
BMSM OLS,0.498282,-0.824445,0.204963,0.601837,-0.772040,0.220156
GARCH,0.903250,NaN,NaN,0.869148,NaN,NaN
GARCH OLS,1.226614,0.388361,0.651075,0.889048,0.051622,0.520579
FIGARCH,1.019155,NaN,NaN,0.943380,NaN,NaN
FIGARCH OLS,1.333614,0.737362,0.769444,0.918354,-0.110252,0.456118


In [27]:
m_z_results_4

{'BMSM': {'alpha_hat': 0.004839419619978563,
  'beta_hat': 0.5970865335293012,
  'alpha_p': 0.005635373552591047,
  'beta_p': 0.0418701639305274},
 'GARCH': {'alpha_hat': 0.006743292234399549,
  'beta_hat': 0.29290050176563204,
  'alpha_p': 6.02146995971295e-07,
  'beta_p': 4.385698315876912e-08},
 'FIGARCH': {'alpha_hat': 0.008194045469497528,
  'beta_hat': 0.16491662798723167,
  'alpha_p': 6.261082265546626e-08,
  'beta_p': 2.221149735519999e-07}}

In [28]:
strategy_4.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.737
Model:                            OLS   Adj. R-squared:                  0.736
Method:                 Least Squares   F-statistic:                     168.0
Date:                Sat, 30 Aug 2025   Prob (F-statistic):          6.66e-105
Time:                        22:18:43   Log-Likelihood:                 27.037
No. Observations:                 821   AIC:                            -44.07
Df Residuals:                     816   BIC:                            -20.52
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1225      0.020     -6.212      0.000      -0.161      -0.084
x1             0.1642      0.027      6.032      0.000       0.111       0.218
x2             0.0562      0.026      2.167      0.030       0.005       0.107
x3             0.1848      0.021      8.782      0.000       0.144       0.226
x4             0.3054      0.014     21.120      0.000       0.277       0.334
==============================================================================
Omnibus:                       63.379   Durbin-Watson:                   0.226
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               29.221
Skew:                           0.265   Prob(JB):                     4.52e-07
Kurtosis:                       2.243   Cond. No.                         3.95
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [29]:
strategy_4.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.627
Model:                            OLS   Adj. R-squared:                  0.625
Method:                 Least Squares   F-statistic:                     117.7
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           2.41e-63
Time:                        22:18:45   Log-Likelihood:                -90.566
No. Observations:                 821   AIC:                             189.1
Df Residuals:                     817   BIC:                             208.0
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.2604      0.024    -11.070      0.000      -0.307      -0.214
x1             0.1111      0.025      4.517      0.000       0.063       0.159
x2             0.2189      0.021     10.285      0.000       0.177       0.261
x3             0.2466      0.021     11.795      0.000       0.206       0.288
==============================================================================
Omnibus:                       37.048   Durbin-Watson:                   0.110
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               18.991
Skew:                           0.181   Prob(JB):                     7.52e-05
Kurtosis:                       2.349   Cond. No.                         1.19
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [30]:
strategy_4.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.706
Model:                            OLS   Adj. R-squared:                  0.705
Method:                 Least Squares   F-statistic:                     266.5
Date:                Sat, 30 Aug 2025   Prob (F-statistic):          1.39e-120
Time:                        22:18:47   Log-Likelihood:                -66.148
No. Observations:                 821   AIC:                             140.3
Df Residuals:                     817   BIC:                             159.1
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0851      0.022     -3.899      0.000      -0.128      -0.042
x1             0.1817      0.023      7.857      0.000       0.136       0.227
x2             0.2742      0.022     12.714      0.000       0.232       0.317
x3             0.3298      0.019     17.188      0.000       0.292       0.367
==============================================================================
Omnibus:                       14.953   Durbin-Watson:                   0.272
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               15.534
Skew:                           0.336   Prob(JB):                     0.000424
Kurtosis:                       2.939   Cond. No.                         1.58
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""